# 📘 目錄

1. [Introduction](#Introduction)
2. [Data Preprocessing](#DP)
3. [Evaluation and Validation](#EaV)
4. [Alternative methods](#AM)
5. [參考文獻](#參考文獻)


<span style='font-size:16px; color:;'>  

# 將 XGBoost 應用於糖尿病預測問題

## 一、Introduction <a id="Introduction"></a>

### 1.1 Problem Background

糖尿病（Diabetes）是全球常見的慢性疾病之一，若未能及早診斷與治療，可能導致心血管疾病、腎臟病與視網膜病變等多種嚴重併發症。因此，如何利用醫療與健康資料建立有效的預測模型，以辨識潛在的高風險族群，已成為醫療資料分析中的重要研究議題。

近年來，**機器學習（Machine Learning）** 方法被廣泛應用於醫療預測問題，例如疾病診斷、風險評估與健康管理。透過分析大量的健康資料，機器學習模型能夠學習不同健康指標與疾病之間的關聯，進而建立具有預測能力的分類模型。

在眾多機器學習方法中，**XGBoost（eXtreme Gradient Boosting）** 是一種基於 Gradient Boosting Decision Tree 的集成學習演算法。由於其具有高效能、良好的泛化能力以及可處理非線性關係等特性，因此在許多分類與預測問題中表現優異。

---

### 1.2 Machine Learning Task

本研究的任務為一個 **二元分類問題（Binary Classification Problem）**：

> 根據個體的健康與生活型態特徵，預測該個體是否患有糖尿病。

在機器學習架構中，可以將此問題表示為：

**Input (Features)**  
每一筆樣本包含多個健康相關特徵，例如：

- 血壓（HighBP）
- BMI
- 吸菸習慣（Smoker）
- 身體活動（PhysActivity）
- 年齡（Age）
- 收入與教育程度等社會經濟指標

這些變數共同形成每個樣本的 **特徵向量**

$$
X = (x_1, x_2, ..., x_p)
$$

**Output (Label)**  

模型的輸出為糖尿病預測結果：

$$
y \in \{0,1\}
$$

其中：

- **0**：未患糖尿病  
- **1**：患有糖尿病  

模型的目標是學習一個函數

$$
f(X) \rightarrow y
$$

使得模型能夠根據輸入特徵準確預測個體的糖尿病狀態。

---

### 1.3 Proposed Method

本研究主要採用 **XGBoost 分類模型** 作為預測方法。XGBoost 是 Gradient Boosting Machine 的改良版本，其核心概念為：

1. 透過多棵決策樹（Decision Trees）逐步學習資料模式  
2. 每一棵新的樹用來修正前一輪模型的預測誤差  
3. 在模型中加入 **L1 與 L2 正則化項**，以降低過度擬合（Overfitting）

相較於傳統的 Gradient Boosting，XGBoost 在演算法與系統層面都進行了優化，因此能夠在大型資料集與高維特徵空間中維持良好的效能。

次要採用另外9種機器學習方法作為對照組，包含  
1.	Linear Discriminant Analysis (LDA)  
2.	Quadratic Discriminant Analysis (QDA)  
3.	LDA with cost-sensitive adjustment  
4.	QDA with cost-sensitive adjustment  
5.	Classification Tree  
6.	Random Forests  
7.	Neural Networks  
8.	Support Vector Machine (SVM)  
9.	Gradient Boosting (GBM)  

    
    
---

### 1.4 Model Evaluation

為了評估模型的預測能力，本研究採用 **Stratified K-fold Cross Validation** 來進行模型訓練與參數選擇。透過將資料分成K組子樣本(並且確保Label 0/1比例與原始資料相同)進行交叉驗證，可以降低模型過度擬合的風險並提供較穩定的模型評估結果。

此外，由於糖尿病資料通常存在 **類別不平衡（Class Imbalance）** 的問題，單純使用準確率（Accuracy）可能無法完整反映模型表現。因此，本研究將使用 **Macro F1-score** 作為主要的模型評估指標，以更公平地評估不同類別的預測效果。

---

### 1.5 Project Objective

本研究的主要目標包括：

- 建立一個以 **XGBoost 為核心的糖尿病預測模型**
- 探討不同模型參數對預測效果的影響
- 透過交叉驗證選擇最佳模型
- 評估模型在不平衡資料情境下的分類表現

最終希望建立一個能夠利用健康與生活型態資料進行 **糖尿病風險預測** 的機器學習模型。

<span style='font-size:16px; color:;'>  


## 二、Data Preprocessing <a id="DP"></a>
   
使用`pandas`讀取資料，總共有$47275$筆資料，計算沒有糖尿病的資料比例，得到訓練資料中沒有糖尿病的患者比例約為$86\%$。  
再來把幾種屬於類別型態的特徵從原始讀取的int64或float64類別，轉換為category資料類別。例如:`HighBP(高血壓), HighChol(高膽固醇)`，這些特徵多為有(1)或沒有(0)的二元特徵。  
我們還展示了敘述統計，連續變數有平均、中位數、標準差等，類別變數則有眾數和頻率等。  
    
最後從訓練集(training dataset)分割出測試集(testing dataset)，其比例為$9:1$(有$4728$筆測試資料)。並且嚴格按照 Label 裡的 0/1 比例來分層隨機抽樣，確保抽出的小樣本，能夠完美保留母體中各個類別的「原始比例」，也就是測試集的Label無糖尿病的比例也是$86\%$
    
最後將Label從訓練集欄位分離出來，分成`X_train`以及`y_train`(轉成整數型態)，並且使用`pd.get_dummies()`)：將類別型態(category)轉換為虛擬變數(dummy variables)，捨棄第一個欄位，避免共線性。

In [2]:
# import pandas as pd
# import numpy as np

# # 1. 讀取資料 (假設 ID 欄位為 index)
# train_data = pd.read_csv("data/train.csv", index_col="ID")
# test_data = pd.read_csv("data/test.csv", index_col="ID")
# submission = pd.read_csv("data/sample_submission.csv")

# # 2. 分離特徵 (X) 與標籤 (y)
# X_train_raw = train_data.drop(columns=['Label'])
# y_train = train_data['Label'].astype(int)
# X_test_raw = test_data.copy()

# # 3. 處理類別變數 (One-Hot Encoding 或虛擬變數)
# # 這裡對應你 R 程式碼中的 model.matrix(~ . -1, data=...)
# # pd.get_dummies 會自動把 object/category 類型的特徵展開
# X_train = pd.get_dummies(X_train_raw, drop_first=True)# drop_first避免共線性
# X_test = pd.get_dummies(X_test_raw, drop_first=True)

# # 確保 test_data 與 train_data 的欄位一致 (有時 test data 會缺少某些類別)
# X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)


In [3]:
import pandas as pd
import numpy as np

# 讀取資料
train_data = pd.read_csv("data/train.csv",index_col='ID')
print(train_data.info())
# 確認 label 的比例
# .value_counts() 會計算各類別的數量，加上 normalize=True 就會變成比例
label_ratio = train_data['Label'].value_counts(normalize=True)
print(f"Label 為 0 的比例: {label_ratio[0]:.4f}")

<class 'pandas.core.frame.DataFrame'>
Index: 47275 entries, 46547 to 14949
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   HighBP                47275 non-null  int64  
 1   HighChol              47275 non-null  float64
 2   CholCheck             47275 non-null  int64  
 3   BMI                   47275 non-null  float64
 4   Smoker                47275 non-null  float64
 5   Stroke                47275 non-null  float64
 6   HeartDiseaseorAttack  47275 non-null  float64
 7   PhysActivity          47275 non-null  int64  
 8   Fruits                47275 non-null  int64  
 9   Veggies               47275 non-null  int64  
 10  HvyAlcoholConsump     47275 non-null  int64  
 11  AnyHealthcare         47275 non-null  int64  
 12  NoDocbcCost           47275 non-null  float64
 13  GenHlth               47275 non-null  float64
 14  MentHlth              47275 non-null  float64
 15  PhysHlth            

In [4]:
print("train_data")
train_data

train_data


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Label
ID,,,,,,,,,,,,,,,,,,,,,
46547,1,0.0,1,28.0,0.0,0.0,0.0,1,0,1,...,0.0,3.0,4.0,2.0,0.0,1,8,4.0,11.0,0.0
22889,0,1.0,1,30.0,1.0,0.0,0.0,1,1,1,...,0.0,2.0,4.0,0.0,0.0,1,4,6.0,9.0,0.0
31776,0,0.0,1,30.0,0.0,0.0,0.0,1,0,1,...,0.0,4.0,0.0,0.0,1.0,1,4,6.0,7.0,0.0
56523,1,0.0,1,31.0,1.0,0.0,0.0,1,1,1,...,0.0,2.0,0.0,2.0,0.0,0,12,5.0,8.0,0.0
55339,0,0.0,1,27.0,0.0,0.0,0.0,1,1,1,...,1.0,3.0,0.0,0.0,0.0,0,10,5.0,5.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4352,0,0.0,1,31.0,0.0,0.0,0.0,1,1,1,...,0.0,1.0,0.0,0.0,0.0,1,5,4.0,10.0,0.0
24379,0,0.0,1,27.0,0.0,0.0,0.0,1,0,0,...,0.0,3.0,6.0,0.0,0.0,0,11,5.0,5.0,0.0
35503,0,0.0,0,30.0,1.0,0.0,0.0,1,0,1,...,0.0,2.0,0.0,0.0,0.0,1,7,4.0,8.0,0.0


In [5]:
# 定義需要轉為類別 (factor) 的欄位
factor_cols = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", 
               "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies", 
               "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", 
               "DiffWalk", "Sex", "Label"]

# 轉換為類別型態 (category)
train_data[factor_cols] = train_data[factor_cols].astype('category')
# 範例寫法：log(1 + train_data[, factor_cols2])
# factor_cols2 = [col for col in train_data.columns if col not in factor_cols + ["MentHlth", "PhysHlth"]]
# train_data[factor_cols2] = np.log1p(train_data[factor_cols2])

In [6]:
# # 讀取時直接將 ID 設為 index
# test_data = pd.read_csv("data/test.csv", index_col="ID")

# # 排除 Label 欄位 (因為 test data 沒有 Label)
# factor_cols_test = [col for col in factor_cols if col != "Label"]

# # 轉換為類別型態
# test_data[factor_cols_test] = test_data[factor_cols_test].astype('category')

# # 讀取 submission 範例
# submission = pd.read_csv("data/sample_submission.csv")

In [7]:
# 加上 include='all' 才會同時顯示連續型變數 (平均值、標準差) 與類別型變數 (頻率、眾數) 的統計量
print(train_data.describe(include='all'))

         HighBP  HighChol  CholCheck          BMI   Smoker   Stroke  \
count   47275.0   47275.0    47275.0  47275.00000  47275.0  47275.0   
unique      2.0       2.0        2.0          NaN      2.0      2.0   
top         0.0       0.0        1.0          NaN      0.0      0.0   
freq    27557.0   28226.0    45554.0          NaN  27937.0  45457.0   
mean        NaN       NaN        NaN     28.94073      NaN      NaN   
std         NaN       NaN        NaN      6.55068      NaN      NaN   
min         NaN       NaN        NaN     13.00000      NaN      NaN   
25%         NaN       NaN        NaN     24.00000      NaN      NaN   
50%         NaN       NaN        NaN     28.00000      NaN      NaN   
75%         NaN       NaN        NaN     32.00000      NaN      NaN   
max         NaN       NaN        NaN     99.00000      NaN      NaN   

        HeartDiseaseorAttack  PhysActivity   Fruits  Veggies  ...  \
count                47275.0       47275.0  47275.0  47275.0  ...   
unique   

In [8]:
from sklearn.model_selection import train_test_split

# 將 train_data 切割為 90% 訓練集 (train_data1) 與 10% 驗證集 (valid_data)
# stratify=train_data['Label'] 代表「請嚴格按照 Label 裡的 0/1 比例來切割」
train_data1, test_data = train_test_split(
    train_data, 
    test_size=0.1,                 # 驗證集佔 10%
    random_state=26,               # 設定亂數種子，確保每次切出來的資料一樣 (等同於 R 的 set.seed)
    stratify=train_data['Label']   # 根據 Label 進行分層隨機抽樣
)

train_data = train_data1.copy()

print(f"訓練集維度: {train_data.shape}")
print(f"測試集維度: {test_data.shape}")

訓練集維度: (42547, 22)
測試集維度: (4728, 22)


In [9]:
# 2. 分離特徵 (X) 與標籤 (y)
X_train_raw = train_data.drop(columns=['Label'])
y_train = train_data['Label'].astype(int)
X_test_raw = test_data.drop(columns=['Label'])
y_test = test_data['Label'].astype(int)


# 3. 處理類別變數 (One-Hot Encoding 或虛擬變數)
# pd.get_dummies 會自動把 object/category 類型的特徵展開
X_train = pd.get_dummies(X_train_raw, drop_first=True)# drop_first避免共線性
X_test = pd.get_dummies(X_test_raw, drop_first=True)


# 確保 test_data 與 train_data 的欄位一致 (有時 test data 會缺少某些類別)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)


In [10]:
X_test.columns

Index(['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income',
       'HighBP_1', 'HighChol_1.0', 'CholCheck_1', 'Smoker_1.0', 'Stroke_1.0',
       'HeartDiseaseorAttack_1.0', 'PhysActivity_1', 'Fruits_1', 'Veggies_1',
       'HvyAlcoholConsump_1', 'AnyHealthcare_1', 'NoDocbcCost_1.0',
       'DiffWalk_1.0', 'Sex_1'],
      dtype='object')

In [11]:
print(f"X_train shape:{X_train.shape}\nX_test shape: {X_test.shape}")
print("X_train")
X_train

X_train shape:(42547, 21)
X_test shape: (4728, 21)
X_train


,BMI,GenHlth,MentHlth,PhysHlth,Age,Education,Income,HighBP_1,HighChol_1.0,CholCheck_1,...,Stroke_1.0,HeartDiseaseorAttack_1.0,PhysActivity_1,Fruits_1,Veggies_1,HvyAlcoholConsump_1,AnyHealthcare_1,NoDocbcCost_1.0,DiffWalk_1.0,Sex_1
ID,,,,,,,,,,,,,,,,,,,,,
1052,36.0,4.0,15.0,20.0,4,4.0,6.0,True,True,True,...,False,False,True,True,True,False,True,False,True,True
19282,27.0,2.0,0.0,0.0,7,5.0,9.0,False,True,True,...,False,False,True,False,True,False,True,False,True,False
57807,33.0,1.0,0.0,0.0,7,6.0,11.0,False,True,True,...,False,False,True,True,True,False,True,False,False,True
58572,29.0,2.0,0.0,30.0,9,6.0,9.0,False,True,True,...,False,False,True,True,True,False,True,False,False,False
27400,29.0,2.0,1.0,0.0,6,6.0,9.0,False,False,True,...,False,False,True,True,True,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14457,24.0,1.0,0.0,0.0,4,6.0,6.0,False,False,True,...,False,False,True,True,True,False,True,False,False,False
54786,31.0,4.0,15.0,2.0,2,4.0,7.0,False,False,True,...,False,False,True,False,True,False,False,True,False,True
55902,22.0,2.0,0.0,4.0,7,6.0,8.0,False,False,True,...,False,False,True,True,True,False,True,False,False,False


<span style='font-size:16px; color:;'>  



## 三、Evaluation and Validation <a id="Eav"></a>



### Stratified K-fold Cross Validation (分層 K 折交叉驗證) 
K 折交叉驗證一種用來評估模型表現或挑選模型參數的驗證程序。它的運作方式首先會將總樣本隨機劃分成 K 個組別（folds）
。接著，演算法會輪流選取其中一個組別作為保留組（holdout group）來進行測試，並將其餘的組別作為訓練模型的資料，這個過程總共會重複進行 K 次
。最後，透過計算模型在各次保留組上的分數（例如f1-score）並進行平均，就能得到模型表現的客觀評估，此方法也常被用於選擇最佳的模型複雜度（例如決策樹的修剪）。分層K折交叉驗證則額外確保每一組別的Label其類別分布與總樣本相近，更適合處理類別不平衡問題。

### Macro F1-score (巨集 F1 分數)
    衡量多類別分類模型表現的指標之一。 F1 分數本身是精確度（Precision）與召回率（Recall）的調和平均數。而 Macro F1-score 的計算方式是先獨立求出每一個單一類別的 F1 分數，接著再將所有的 F1 分數直接進行算術平均。這種指標的特點在於它賦予所有類別完全相同的權重，不會去考慮各類別在資料中的樣本數量多寡，因此特別適合用來評估資料存在「類別不平衡」時，模型對於少數類別的預測表現。以下是計算的流程：
    
#### 1. 混淆矩陣 (Confusion Matrix)

| | 預測為正類 (Predicted 1) | 預測為負類 (Predicted 0) |
| :--- | :--- | :--- |
| **實際為正類 (Actual 1)** | **TP** (True Positive) | **FN** (False Negative) |
| **實際為負類 (Actual 0)** | **FP** (False Positive) | **TN** (True Negative) |

---

#### 2. 基礎評估指標 (Precision, Recall, F1-score)

**精確率 (Precision)** ：模型預測為正類的樣本中，實際為正類的比例。
$$\text{Precision} = \frac{TP}{TP + FP}$$

**召回率 (Recall)** ：實際為正類的樣本中，被模型成功預測出來的比例。
$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1-score**：精確率與召回率的調和平均數，用於綜合評估單一類別的表現。
$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

---

#### 3. 計算 Macro F1-score

將類別 0 與類別 1 各自的 F1-score 算出來後，取簡單算術平均，讓大類別與小類別擁有相同的評估權重。
$$\text{Macro F1} = \frac{\text{F1}_{\text{Class 0}} + \text{F1}_{\text{Class 1}}}{2}$$


In [30]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import numpy as np


<span style='font-size:16px; color:;'>  



## 四、Alternative Method <a id="AM"></a>

    
    
在進入到主要方法的XGBoost之前，我們先簡單介紹其他的方法，以方便和主要方法做比較，詳細可以參考Johnson and Wichern (2002)的教科書。   
    
### 1. 線性判別分析 (Linear Discriminant Analysis, LDA)
Linear Discriminant Analysis (LDA) 是一種分類方法，通常基於各個母體呈現多變量常態分配且具有**相同共變異數矩陣**$(\Sigma_1=\Sigma_2=\Sigma)$的假設
。此方法會計算線性判別分數，透過對變數進行線性組合，尋找能讓組間變異相對於組內變異達到最大化的分類邊界
。相較於二次判別分析，LDA 對於違反常態分配假設的情況具有較高的穩健性


In [43]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 1. 建立 LDA 模型實例
# 在 scikit-learn 中，LDA 預設就會使用 SVD 求解，不需特別設定公式
lda = LinearDiscriminantAnalysis()

# 2. 設定 10-Fold 分層交叉驗證
# n_splits=10 切成 10 份，shuffle=True 打亂資料確保隨機性
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=26)

# 3. 執行交叉驗證並計算 Macro F1-score
# scoring='f1_macro' 直接指定我們想看的評估指標

# cross_val_score 只能用「0.5」作為預測機率的決策閥值（Threshold）
cv_scores = cross_val_score(
    lda, 
    X_train, 
    y_train, 
    cv=cv_strategy, 
    scoring='f1_macro',
#     scoring='roc_auc'
)

# 輸出 K-fold 的驗證結果
print(f"LDA 10-fold Cross Validation Macro F1 score: \n{np.round(cv_scores, 4)}")
print(f"LDA average Macro F1 score (CV Mean): {cv_scores.mean():.5f}")

# 測試資料結果
lda.fit(X_train,y_train)
y_pred = lda.predict(X_test)
final_scores = f1_score(y_test,y_pred,average='macro')
print("\n最終測試結果（使用獨立測試集）:")
print(f"測試集 Macro F1: {final_scores:.5f}")

LDA 10-fold Cross Validation Macro F1 score: 
[0.5951 0.6021 0.596  0.6223 0.6099 0.5933 0.5969 0.6052 0.5888 0.5888]
LDA average Macro F1 score (CV Mean): 0.59983

最終測試結果（使用獨立測試集）:
測試集 Macro F1: 0.61786


<span style='font-size:16px; color:;'>  

    
### 2. 二次判別分析 (Quadratic Discriminant Analysis, QDA)
Quadratic Discriminant Analysis (QDA)適用於各個母體呈現多變量常態分配，但**共變異數矩陣不相同** $(\Sigma_1 \neq \Sigma_2)$的情境。因為各組具有獨立的共變異數矩陣，QDA 的分類區域是由觀測變數的二次函數所定義，並透過計算二次判別分數來進行分類。需要特別注意的是，QDA 對於違反常態分配假設非常敏感，如果資料並非來自多變量常態分配，使用二次法則可能會導致非常高的誤判率
   

In [44]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda = QuadraticDiscriminantAnalysis()
cv_scores = cross_val_score(
    qda,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=26),
    scoring='f1_macro'
)

# 輸出 K-fold 的驗證結果
print(f"QDA 10-fold Cross Validation Macro F1 score: \n{np.round(cv_scores, 4)}")
print(f"QDA average Macro F1 score (CV Mean): {cv_scores.mean():.5f}")


# 測試資料結果
qda.fit(X_train,y_train)
y_pred = qda.predict(X_test)
final_scores = f1_score(y_test,y_pred,average='macro')
print("\n最終測試結果（使用獨立測試集）:")
print(f"測試集 Macro F1: {final_scores:.5f}")


QDA 10-fold Cross Validation Macro F1 score: 
[0.6394 0.6417 0.6204 0.6263 0.6419 0.6227 0.6297 0.6261 0.6317 0.6314]
QDA average Macro F1 score (CV Mean): 0.63113

最終測試結果（使用獨立測試集）:
測試集 Macro F1: 0.63783


<span style='font-size:16px; color:;'>  

### 3. 具成本敏感調整的 LDA (LDA with cost-sensitive adjustment)
LDA with cost-sensitive adjustment旨在解決不同誤判情況帶有不同嚴重性（即誤判成本）的實務問題。一般的決策通常只求最小化總誤判機率，但此方法進一步考量各母體的事前機率與誤判成本，目標是**最小化預期誤判成本 (Expected Cost of Misclassification, ECM)**
。在共變異數矩陣相等的假設下，該法則會在線性判別的計算中引入誤判成本比例與事前機率比例的對數值（即 
$$    
\ln\left[\left(\frac{c[1|2]}{c[2|1]}\right)\left(\frac{p2}{p1}\right)\right]
$$
作為決策門檻，藉此調整線性決策邊界，使模型更傾向避免發生代價高昂的誤判  

當漏抓糖尿病（False Negative, 實際為 1 預測為 0）的代價是 5 倍時，我們在數學上可以透過人為地「放大」類別 1 的先驗機率，來逼迫 LDA 的決策邊界向類別 0 退縮，寧可多錯殺（False Positive），也要盡量把有病的人抓出來。

In [45]:
# 1. 計算原始的先驗機率 (Original Priors)
# 使用 y_train 算出 0 和 1 的真實比例
prior_0 = (y_train == 0).mean()
prior_1 = (y_train == 1).mean()

# 2. 定義錯誤成本 (Cost) 並調整先驗機率
cost_FP = 1  # 實際沒病(0)預測成有病(1)的代價
cost_FN = 5  # 實際有病(1)預測成沒病(0)的代價 (設定為 5 倍)

adj_prior_0 = prior_0 * cost_FP
adj_prior_1 = prior_1 * cost_FN

# 正規化，讓調整後的先驗機率相加為 1
total_prior = adj_prior_0 + adj_prior_1
adj_prior_0 = adj_prior_0 / total_prior
adj_prior_1 = adj_prior_1 / total_prior

print(f"原始先驗機率 -> Label 0: {prior_0:.4f}, Label 1: {prior_1:.4f}")
print(f"加入Cost後的先驗機率 -> Label 0: {adj_prior_0:.4f}, Label 1: {adj_prior_1:.4f}\n")

# 3. 建立 LDA 模型實例，並放入自訂的 priors
lda_cost = LinearDiscriminantAnalysis(priors=[adj_prior_0, adj_prior_1])

# 4. 設定 10-Fold 分層交叉驗證
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=26)

# 5. 執行交叉驗證並計算 Macro F1-score
cv_scores = cross_val_score(
    lda_cost, 
    X_train, 
    y_train, 
    cv=cv_strategy, 
    scoring='f1_macro'
)

# 輸出 K-fold 的驗證結果
print(f"LDA (Cost=5) 10-fold CV Macro F1 score: \n{np.round(cv_scores, 4)}")
print(f"LDA (Cost=5) average Macro F1 score (CV Mean): {cv_scores.mean():.5f}")

# 測試資料結果
lda_cost.fit(X_train,y_train)
y_pred = lda_cost.predict(X_test)
final_scores = f1_score(y_test,y_pred,average='macro')
print("\n最終測試結果（使用獨立測試集）:")
print(f"測試集 Macro F1: {final_scores:.5f}")


原始先驗機率 -> Label 0: 0.8580, Label 1: 0.1420
加入Cost後的先驗機率 -> Label 0: 0.5471, Label 1: 0.4529

LDA (Cost=5) 10-fold CV Macro F1 score: 
[0.6605 0.6527 0.6347 0.6522 0.6626 0.6505 0.6272 0.6476 0.6591 0.648 ]
LDA (Cost=5) average Macro F1 score (CV Mean): 0.64951

最終測試結果（使用獨立測試集）:
測試集 Macro F1: 0.65563


<span style='font-size:16px; color:;'>  

### 4. 具成本敏感調整的 QDA (QDA with cost-sensitive adjustment) 
QDA with cost-sensitive adjustment則是同時結合了共變異數矩陣不相等以及最小化預期誤判成本 (ECM) 兩項考量的方法
。在計算複雜的二次決策區域$R_1$與$R_2$時，除了使用各自獨立的共變異數矩陣外，決策規則同樣會將**誤判成本比值與事前機率比值的對數**納入比較門檻中
。這樣的調整會改變由二次函數構成的分類邊界，讓模型在適應各組資料變異程度不同的同時，也能達到降低整體誤判損失的最佳化結果
    

In [46]:
# 這次強制調整完(加入cost)的先驗機率為0.5:0.5

qda_default = QuadraticDiscriminantAnalysis()
qda_default.fit(X_train, y_train)
prior_0, prior_1 = qda_default.priors_
cost = prior_0/prior_1
print(f'模型先驗機率：{prior_0,prior_1}')
print(f'cost應設倍數:{cost}')

qda_cost = QuadraticDiscriminantAnalysis(priors=[0.5,0.5])
cv_scores = cross_val_score(
    qda_cost,
    X_train,
    y_train,
    cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=26),
    scoring = 'f1_macro'
)

print(f"QDA (Cost={cost:.0f}) 10-fold CV Macro F1 score: \n{np.round(cv_scores, 4)}")
print(f"QDA (Cost={cost:.0f}) average Macro F1 score (CV Mean): {cv_scores.mean():.5f}")

# 測試資料結果
qda_cost.fit(X_train,y_train)
y_pred = qda_cost.predict(X_test)
final_scores = f1_score(y_test,y_pred,average='macro')
print("\n最終測試結果（使用獨立測試集）:")
print(f"測試集 Macro F1: {final_scores:.5f}")

模型先驗機率：(0.8579688344654147, 0.14203116553458528)
cost應設倍數:6.040708257488003
QDA (Cost=6) 10-fold CV Macro F1 score: 
[0.5949 0.6036 0.5788 0.5822 0.5914 0.5919 0.6002 0.5874 0.6021 0.5959]
QDA (Cost=6) average Macro F1 score (CV Mean): 0.59285

最終測試結果（使用獨立測試集）:
測試集 Macro F1: 0.60414


<span style='font-size:16px; color:;'>  

    
### 5. 分類樹 (Classification Tree)
Classification Tree是一種將特徵空間透過遞迴二元分割來劃分為多個區域的機器學習模型
。當目標變數為類別型態時，分類樹會根據節點的不純度指標（例如吉尼指數 Gini index 或交叉熵 cross-entropy）來尋找最佳的分割變數與切分點，並以每一個劃分區域中出現比例最高的多數類別（majority class）作為該區域的預測結果
。為了避免模型對訓練資料過度配適，通常會先建立一棵完整的大樹，接著再利用誤判率（misclassification error rate）來進行成本複雜度修剪（cost-complexity pruning），以找出架構最合適且預測能力最佳的子樹，  
    **分類樹(classification tree)** 完整流程如下：
    

1. **生長大樹 (Tree Growing)** ：   
    從包含所有訓練資料的根節點開始，透過遞迴二元分割，在每個節點尋找能讓**「吉尼指數」或「交叉熵」最小化（即純度最大化）的變數與切分點
。這個分割過程會不斷重複，直到每一個終端節點（葉節點）包含的樣本數少於設定的最低下限 (min_size)** ，藉此生成一棵不限制大小的完整大樹
。  
    
2. **成本複雜度修剪 (Cost-complexity Pruning)** ： 因為完整大樹容易過度配適，接著會引入調節參數 α 來控制樹的複雜度。較大的 α 會對節點數量（|T|）過多的樹施加較重的懲罰。演算法會透過砍掉（合併）對降低誤差貢獻度最低的節點，產生一系列對應不同 α 值的候選子樹。在此階段，分類樹是使用「誤判率 (Misclassification error)」來計算各節點的不純度 (Node Impurity)，並將其與懲罰項 α∣T∣ 結合，藉此計算出整棵候選子樹的成本複雜度 $C_{\alpha}(T)$ 來進行修剪
3. **K 折交叉驗證 (K-fold Cross-validation)** ：   
    最後，我們利用 K 折交叉驗證來測試不同的 α 值。對於每一個 α，計算其對應子樹在驗證集上的**「平均誤判率」**。我們挑選出能讓平均誤判率最小的 α 值，並輸出該 α 對應的子樹，這就是最終形成的分類樹模型  
    
補充與**迴歸樹(regression tree)** 之比較：
    
1. **節點的預測結果 (Prediction)**  
* 分類樹 ： 預測該區域內出現比例最高的類別（多數決）。  
* 迴歸樹 ： 預測該區域內所有訓練資料目標變數$y_i$的平均值 (average/mean)。假設節點m代表區域$R_m$，期預測常數$\hat{c}_m$即為$mean(y_i|x_i\in R_m)$
    
    
2. **生長大樹的分支指標 (Splitting Criterion)**  
* 分類樹： 尋找能讓「吉尼指數」或「交叉熵」最小化（純度最大化）的切分點。
* 迴歸樹： 尋找能讓**「殘差平方和 (Residual Sum of Squares, RSS)」最小化**的切分點
。演算法會評估所有的預測變數與切分點，目標是讓分割出來的兩個新區域中，觀測值與該區域平均值之間的平方誤差總和達到最小

3. **成本複雜度修剪的指標 (Cost-complexity Pruning)**
* 分類樹： 計算節點不純度時使用的是「誤判率 (Misclassification error)」，也就是$\frac{1}{N_m} \sum_{x_i\in R_m}I(y_i\ne k(m))$。剛好等同於 1 減去多數類別在該節點中所佔的比例
* 迴歸樹： 節點的不純度指標$Q_m(T)$ 改為使用該節點內的均方誤差 (Mean Squared Error)，也就是$\frac{1}{N_m}\sum_{x_i\in R_m} (y_i - \hat{c}_m)^2$。當計算整棵樹的成本複雜度$C_\alpha(T)$時，整體考量的就是修剪樹的總 RSS 加上對節點數量的懲罰項 α∣T∣
    
4. **K 折交叉驗證的評估標準 (K-fold Cross-validation)**  
* 分類樹： 挑選能讓驗證集「平均誤判率」最小的 α 值。
* 迴歸樹： 針對每一個 α 值，計算其在驗證集上的 RSS，最後挑選出能讓**「平均 RSS (Average RSS)」最小的 α**，並輸出對應的最佳修剪子樹

In [54]:
# 只生長樹，不修剪的版本。
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# 1. 建立分類樹模型實例
# minsplit=20每一個末端節點最多只有20個樣本
# (註: R 的 cp 參數對應 Python 的 ccp_alpha，若需要剪枝可以另外設定)
tree_model = DecisionTreeClassifier(min_samples_split=20, random_state=26)

# 2. 設定 10-Fold 分層交叉驗證
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=26)

# 準備一個空列表來接每次 fold 的分數
macro_f1_scores = []

# 設定你的自訂閥值
threshold = 0.60

# 3. 手動展開 K-fold 迴圈
for fold, (train_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), 1):# start=1
    
    # 根據 index 切分出這一回合的訓練集與驗證集
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[valid_idx], y_train.iloc[valid_idx]
    
    # 訓練模型
    tree_model.fit(X_tr, y_tr)
    
    # 取得預測機率 (回傳矩陣的 shape 為 [樣本數, 2])
    # [:, 0] 代表所有樣本屬於類別 0 的機率，[:, 1] 則是屬於類別 1 的機率
    y_val_probs = tree_model.predict_proba(X_val)
    
    # 套用自訂邏輯：對應你的 if (one_row["0"] > 0.76) return "0" else return "1"
    # np.where(條件, 條件成立給的值, 條件不成立給的值)
    y_pred_custom = np.where(y_val_probs[:, 0] > threshold, 0, 1)
    
    # 計算 Macro-F1
    macro_f1 = f1_score(y_val, y_pred_custom, average='macro')
    macro_f1_scores.append(macro_f1)
    
#     print(f"Fold {fold} Macro-F1: {macro_f1:.4f}")

# 輸出總結
print("-" * 30)
print(f"分類樹(未修剪) (Threshold={threshold}) 10-fold CV Macro F1 score: \n{np.round(macro_f1_scores, 4)}")
print(f"分類樹(未修剪) (Threshold={threshold}) average Macro F1 score (CV Mean): {np.mean(macro_f1_scores):.5f}")

# 測試資料結果
tree_model.fit(X_train,y_train)
y_pred = tree_model.predict(X_test)
final_scores = f1_score(y_test,y_pred,average='macro')
print("\n最終測試結果（使用獨立測試集）:")
print(f"測試集 Macro F1: {final_scores:.5f}")



------------------------------
分類樹(未修剪) (Threshold=0.6) 10-fold CV Macro F1 score: 
[0.6175 0.6285 0.5969 0.6156 0.6044 0.6029 0.6015 0.5956 0.6059 0.6163]
分類樹(未修剪) (Threshold=0.6) average Macro F1 score (CV Mean): 0.60851

最終測試結果（使用獨立測試集）:
測試集 Macro F1: 0.59905


In [29]:
from sklearn.metrics import f1_score, make_scorer

def macro_f1_threshold(y_true,y_probs,threshold=0.6):
    y_pred = np.where((1-y_probs)>threshold,0,1)
    return f1_score(y_true, y_pred, average='macro')

# 把它包裝成 cross_val_score 認得的 Scorer (需要指定 needs_proba=True)
my_scorer = make_scorer(macro_f1_threshold, response_method='predict_proba',threshold=0.6)

cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=26)
cv_scores = cross_val_score(tree_model, X_train, y_train, cv=cv_strategy, scoring=my_scorer)
print(cv_scores.mean())


0.6085073560726459


In [67]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, make_scorer

# ==========================================
# 1. 取得 alpha 候選清單 (先讓樹長大來探路)
# ==========================================
mss = 20

clf_path = DecisionTreeClassifier(min_samples_split=mss, random_state=26)
path = clf_path.cost_complexity_pruning_path(X_train, y_train)
all_alphas = path.ccp_alphas

# ==========================================
# 2. 模擬 rpart 的高效抽樣：只取約 20 個具代表性的 alpha
# ==========================================
# 扣除最後一個 (會把樹剪光變成只有根節點的 alpha)
all_alphas = all_alphas[:-1]

num = 10
if len(all_alphas) > num:
    # 計算步伐大小，均勻抽出 num 個關鍵的 alpha 節點
    step = len(all_alphas) // num
    sampled_alphas = all_alphas[::step] 
else:
    sampled_alphas = all_alphas

print(f"原先產生了 {len(all_alphas)} 個 alpha。")
print(f"優化後，我們只選取 {len(sampled_alphas)} 個最具代表性的 alpha 交給 C 引擎運算。")

# ==========================================
# 3. 定義 GridSearchCV (取代手動 for 迴圈)
# ==========================================
# 將我們要測試的參數包裝成字典
param_grid = {'ccp_alpha': sampled_alphas}

# 沿用你原本 10-fold 分層交叉驗證的設定
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=26)

# 建立一個基礎決策樹
base_tree = DecisionTreeClassifier(min_samples_split=mss, random_state=26)

# 🌟 核心加速區：封裝進 GridSearchCV：scikit-learn 中用於超參數調優的工具，會自動嘗試所有參數組合，並透過交叉驗證找出最佳的超參數設定
# scoring='accuracy' 對應你 R 程式碼中的「尋找最小誤判率」
my_scorer = make_scorer(macro_f1_threshold, response_method='predict_proba',threshold=0.6)

grid_search = GridSearchCV(
    estimator=base_tree,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring=my_scorer,#'accuracy'
#     scoring='accuracy',
    n_jobs=-1,           # 關鍵！-1 代表呼叫你電腦「所有的 CPU 核心」同時平行運算
    verbose=1            # 顯示進度條，讓你知道它正在飛速運作
)

# ==========================================
# 4. 啟動高速訓練與尋優
# ==========================================
grid_search.fit(X_train, y_train)

# 印出最佳參數
print("-" * 30)
print(f"最佳懲罰係數 (Alpha): {grid_search.best_params_['ccp_alpha']:.6f}")
# accuracy 越高，代表誤判率越低
print(f"最高平均macro-f1: {grid_search.best_score_:.4f}") 


# ==========================================
# 5. 取得最佳模型與結果
# ==========================================
# grid_search.best_estimator_ 會直接回傳那棵「贏得比賽、設定好最佳 alpha」的決策樹
best_tree = grid_search.best_estimator_

cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=26)
macro_f1_scores = cross_val_score(best_tree, X_train, y_train, cv=cv_strategy, scoring=my_scorer)


# 輸出總結
print("-" * 30)
print(f"分類樹(ccp修剪) 10-fold CV Macro F1 score: \n{np.round(macro_f1_scores, 4)}")
print(f"分類樹(ccp修剪) average Macro F1 score (CV Mean): {np.mean(macro_f1_scores):.5f}")

# 測試資料結果
best_tree.fit(X_train,y_train)
y_pred = best_tree.predict(X_test)
final_scores = f1_score(y_test,y_pred,average='macro')
print("\n最終測試結果（使用獨立測試集）:")
print(f"測試集 Macro F1: {final_scores:.5f}")




原先產生了 1198 個 alpha。
優化後，我們只選取 11 個最具代表性的 alpha 交給 C 引擎運算。
Fitting 10 folds for each of 11 candidates, totalling 110 fits
------------------------------
最佳懲罰係數 (Alpha): 0.000057
最高平均macro-f1: 0.6163
------------------------------
分類樹(ccp修剪) 10-fold CV Macro F1 score: 
[0.6279 0.6151 0.6292 0.6204 0.6285 0.6012 0.604  0.6029 0.6084 0.6257]
分類樹(ccp修剪) average Macro F1 score (CV Mean): 0.61633

最終測試結果（使用獨立測試集）:
測試集 Macro F1: 0.61679


<span style='font-size:16px; color:;'>  

程式碼首先讓決策樹充分生長，捕捉到 1198 個候選的懲罰係數（$\alpha$），接著巧妙地均勻抽樣出 11 個最具代表性的 $\alpha$ 值，並搭配自訂的 Macro-F1 評分器（固定閥值為 0.6）與 GridSearchCV 進行 10 折交叉驗證。這不僅大幅節省了運算時間，更精準找出了最佳的懲罰係數 0.000057。從執行結果來看，這是一次非常成功的模型優化！初級版（未修剪）的決策樹在交叉驗證時分數為 0.6085，但面對未知的獨立測試集時掉到了 0.5990，顯示模型死背了訓練集的雜訊，出現了典型的「過度配適（Overfitting）」現象。相反地，專業修剪版不僅將交叉驗證分數提升至 0.6163，在獨立測試集上更穩定維持在 0.6167。這完美證明了經過 $\alpha$ 剪枝後的決策樹，成功剔除了冗餘的節點，保留了最具泛化能力的預測規則，在未知資料上的表現既準確又穩健。

<span style='font-size:16px; color:;'>  

 
### 6. 隨機森林 (Random Forests) 
Random Forests是一種基於裝袋法（Bagging）改良的集成學習方法，主要目的是為了解決單一決策樹容易產生高變異性的問題
。該方法會利用拔靴法（bootstrap）從原始資料中抽出多組訓練樣本，並為每一組樣本建立一棵不進行修剪的完整決策樹
。隨機森林最核心的設計在於「隨機選取特徵」：在每次進行節點分裂時，演算法只會從所有的預測變數中隨機挑選一小部分來尋找最佳切分點，這個機制能大幅降低各棵決策樹之間的相關性，進一步提升整體模型平均後的變異縮減效果
。在面對分類問題時，一筆新資料的最終預測結果是由森林中所有的決策樹共同進行多數決（majority vote）來決定
    
    

<span style='font-size:16px; color:;'>  

### 7. 類神經網路 (Neural Networks, NN)
Neural Networks是一種模擬人類大腦行為的機器學習演算法，可作為兩階段的分類或迴歸模型來使用
。其核心概念是將輸入變數進行線性組合，藉此萃取出隱藏層的衍生特徵，接著再透過非線性函數（例如 tanh）將這些特徵進行轉換，以建立預測目標變數的模型
。這類模型通常被稱為前饋類神經網路 (feedforward neural networks) 或多層感知器 (MLPs)，因為資訊在網路中的流向是單向的：從輸入層出發，經過隱藏層的特徵萃取與轉換，最終抵達輸出層
。在模型的參數訓練上，主要是運用反向傳播演算法 (Back-propagation)，透過微積分的連鎖律來計算各層參數的梯度，並結合梯度下降法來不斷更新權重，藉此達到最小化整體模型預測誤差的目標
    

<span style='font-size:16px; color:;'>  

### 8.支持向量機 (Support Vector Machine, SVM)
SVM是一種基於統計學習理論的分類演算法
。它的核心概念是尋找一個具有**最大邊界 (maximum margin)** 的線性決策超平面，讓分類邊界與最靠近的資料點（即支持向量）保持最遠的距離，這種設計使得模型對離群值具有較高的穩健性，並具備強大的泛化能力
。當面對線性不可分或包含雜訊的資料時，SVM 可以引入寬鬆變數 (slack variables) 來建立**軟邊界 (soft margin)**
，或是利用**核技巧 (kernel trick)** 將原始資料對應到更高維度的特徵空間，使其在該空間中變得線性可分
。此外，雖然 SVM 最初是為了二元分類設計，但透過一對多 (one-against-all) 或一對一 (one-against-one) 等策略，也能有效處理多類別的分類問題



<span style='font-size:16px; color:;'>  

### 9.梯度提升機 (Gradient Boosting, GBM)
GBM是一種循序式 (sequential) 的集成學習方法，主要透過結合多個弱學習器（通常是決策樹）來建立強大的預測模型
。它的訓練過程類似於數值最佳化中的梯度下降法：模型會透過前向逐步 (forward stagewise) 的方式建立，在每一次疊代中，演算法會計算當前模型損失函數的**負梯度 (negative gradient)** （在平方誤差損失下即等同於模型殘差），並訓練一棵新的決策樹來配適這個負梯度，接著將其加入總模型中不斷修正先前的錯誤。為了避免模型過度配適 (overfitting)，GBM 通常會搭配多種正規化策略，例如限制決策樹的大小、使用**縮減機制 (shrinkage)** 來控制學習率，或是採用**子抽樣 (subsampling)** 技術來進一步提升預測的準確度與運算效率

<span style='font-size:16px; color:;'>  

參考文獻:  

Johnson, R. A., & Wichern, D. W. (2002). Applied multivariate statistical analysis.